In [0]:
table_name = "dim_network_status"
silver_table_fqn = f"he_prod_incen_analytics_dbw_01.silver.{table_name}"

df = spark.read.table("he_prod_incen_analytics_dbw_01.bronze.dim_network_status")

In [0]:
from pyspark.sql.functions import col, upper, trim, when, abs as spark_abs

# 1. Standardize Text & Safe Cast ID
df = df.withColumn("network_status_id", upper(trim(col("network_status_id")))) # Keeping as string in case it's text
df = df.withColumn("network_status_name", upper(trim(col("network_status_name"))))

# 2. Precision Optimization
is_valid_coin = trim(col("coinsurance_percentage")).rlike("^[+-]?([0-9]+\\.?[0-9]*|[0-9]*\\.?[0-9]+)$")
df = df.withColumn("coinsurance_percentage", when(is_valid_coin, trim(col("coinsurance_percentage")).cast("decimal(5,2)")).otherwise(None))

is_valid_ded = trim(col("deductible_applied")).rlike("^[+-]?([0-9]+\\.?[0-9]*|[0-9]*\\.?[0-9]+)$")
df = df.withColumn("deductible_applied", when(is_valid_ded, trim(col("deductible_applied")).cast("decimal(15,2)")).otherwise(None))
df = df.withColumn("deductible_applied", spark_abs(col("deductible_applied")))

# 3. Boolean Mapping
df = df.withColumn("prior_auth_required", 
                   when(upper(trim(col("prior_auth_required"))).isin("YES", "1", "TRUE"), True)
                   .when(upper(trim(col("prior_auth_required"))).isin("NO", "0", "FALSE"), False)
                   .otherwise(None).cast("boolean"))

# 4. Drop audit/source SKs
df = df.drop("_ingested_at", "_source_file", "network_status_sk")

display(df.limit(5))

In [0]:
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table_fqn)
print(f"✅ Successfully wrote {table_name} to Silver layer.")